# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *70* |
| **Integrantes** | *Isabela Cuartas Vence · Juan Camilo Gomez Murillo* |
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 23 de agosto |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*¿Qué se quiere resolver y por qué importa? Máximo tres párrafos.
Debe quedar claro qué preguntas del negocio deberá responder esta base de datos.*

Wanderbricks es una plataforma de alquiler de vacaciones donde los anfitriones pueden listar sus propiedades y los usuarios pueden hacer reservas que luego pueden ser modificadas, confirmadas o canceladas. El núcleo del problema empresarial no se limita a conocer la cantidad de reservas, sino a comprender su evolución con qué frecuencia se producen cambios de estado, cuánto tiempo pasa desde la creación de una reserva hasta su actualización, y si este comportamiento difiere según el país del usuario o de la propiedad. 

La tabla booking_updates es la que hace visible esta dinámica temporal, mientras que users y countries permiten segmentar ese comportamiento por perfil de usuario y por geografía. Esto es relevante ya que el equipo de operaciones y el equipo comercial de Wanderbricks requieren decisiones fundamentadas en datos, no en suposiciones identificar mercados con alta tasa de cancelación para modificar políticas de reembolso, y detectar si ciertos comentarios de los usuarios provocan más cambios de última hora y permiten anticipar picos de demanda por región. Sin una base de datos que conserve el historial de cambios de cada reserva , estas preguntas no se pueden responder, porque la tabla bookings por sí sola solo muestra una fotografía del presente, no la trayectoria de cada reserva.
 
Esta base de datos debe proporcionar respuestas a preguntas como: ¿cuál es la tasa de cancelación o modificación de reservas en cada país?, ¿cuánto tiempo, en promedio, transcurre entre la creación de una reserva y su primera actualización?, ¿qué países tienen la mayor cantidad de usuarios activos y cómo se relaciona esto con el número de reservas confirmadas?, ¿hay patrones de comportamiento que ayuden a predecir cancelaciones?, y ¿cómo ha variado el volumen de reservas y actualizaciones a lo largo del tiempo según la región geográfica?

---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

In [0]:
# TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)
from pyspark.sql import functions as F

tablas = ["users", "countries", "bookings", "booking_updates"]

for t in tablas:
    print(f"\n--- Nulos en {t} ---")
    df = spark.table(f"samples.wanderbricks.{t}")
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

#Cardinalidades (columna por columna, para cada tabla):

for t in tablas:
    print(f"\n--- Cardinalidad en {t} ---")
    df = spark.table(f"samples.wanderbricks.{t}")
    total = df.count()
    df.select([F.countDistinct(F.col(c)).alias(c) for c in df.columns]).display()
    print(f"(Total de filas para comparar: {total:,})")

In [0]:
# Esquema y tipos de cada tabla
for t in tablas:
    print(f"\n--- Esquema de {t} ---")
    spark.table(f"samples.wanderbricks.{t}").printSchema()

Relaciones entre tablas: 
bookings.user_id → users.user_id (N:1): cada reserva pertenece a un usuario.

booking_updates.booking_id → bookings.booking_id (N:1): cada reserva puede tener cero o varias actualizaciones (83,068 actualizaciones sobre 72,247 reservas).

users.country → countries.country (N:1): la relación geográfica se hace por nombre de país, no por country_code, aunque ambas columnas existen en countries.

Esto es un hallazgo de calidad de datos a tener en cuenta: exige coincidencia exacta de string (sensible a mayúsculas/tildes/espacios), lo cual valida antes del join.

Observación sobre booking_updates
El esquema de booking_updates replica casi por completo el de bookings (check_in, check_out, guests_count, status, total_amount), agregando booking_update_id y updated_at. Esto confirma que no es una tabla relacional tradicional de "detalle", sino un log de cambios, cada fila representa el 
estado de una reserva en un momento dado. Este patrón encaja naturalmente con la capacidad de time travel de Delta Lake, que exploraremos en la sección 4.5.

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*



Criterio del caso: Relacion users - bookings (1 usuario -> varias reservas, cardinalidad 54708 usuarios / 72247 reservas)  

Relacional: Queda exacto con las llaves foraneas nativas para este tipo de relacion de 1 a muchos 

NOSQL: Se tendria que hacer un duplicado entre cada reserva o pasar directamente a hacer unas busquedas manuales

Lakehouse: Es igual a una base de datos relacional, se hace via JOIN sobre unas tablas DELTA 

Decision: Lakehouse tiene un mismo beneficio, sin tener que perder una flexibilidad futura

____________________________________________________________________________________________________________________________

Criterio del caso: Integridad de llaves (booking_id y user_id no tiene duplicados resaltados en la seccion 2)

Relacional: constraints y llaves primarias que garantizan una unidad 

NOSQL: La mayoria de NOSQL no valida unidad de forma nativa

Lakehouse: Se puede confirmar con MERGE / dropduplicates, aunque no hay un constraint automatizado como en relacional exacto

Decision: Relacional gana en este ambito puntual, pero lakehouse, mejora muchisimo mas con la compensa con control programatico

____________________________________________________________________________________________________________________________

Criterio del caso: Confirmacion del historial de cambios (7943 duplicados que han sido detectados en booking_updates y evolucionan en estados de reserva)

Relacional: No existe forma nativa de ir a una version anterior de la tabla sin diseñar una auditoria manual

NOSQL: No tiene una version nativamente

Lakehouse: "DESCRIBE HISTORY" y "VERSION AS OF" sin diseño adicional

Decision: Lakeouse (Ventaja abismal)

____________________________________________________________________________________________________________________________

Criterio del caso: Transacciones atomicas al actualizar reservas (MERGE de "booking_updates" hacia "bookings")

Relacional: Soporta ACID de forma efectiva con los motores que tiene como PostgreSQL

NOSQL: La mayoria de NoSQL sacrifica a ACID por escalabilidad (Exceptuando casos muy especificos)

Lakehouse: ACID completo via DeltaLake, igual de relleno y robusto que el relacional

Decision: Diriamos que es un empate puesto que entre relacional y lakehouse, gana lakehouse por sumar todo lo anterior

____________________________________________________________________________________________________________________________

Criterio del caso: Escalabilidad y datos futuros mixtos

RelacionaL: Agregar esa tabla obligaria a tener que normalizar los arrays en multiples tablas entrelazadas

NOSQL: En ese ambito es flexible pero perderia la consistencia que ya tienen sus tablas transaccionales

Lakehouse: Combina la flexibilidad para clickstream sin perder ACID en bookings y users

Decision: Lakehouse (Por motivos principales)

____________________________________________________________________________________________________________________________

Explicacion:

Si bien las cuatro tablas examinadas son completamente tabulares y encajarían en un modelo relacional puro, el caso de Wanderbricks en su totalidad presenta datos de navegación con estructuras anidadas que un modelo relacional podría manejar con dificultad.
Por eso se optó por un lakehouse, conserva las garantías de integridad y las relaciones llave-foránea del modelo relacional , pero añade capacidades de versionado  y flexibilidad para datos semiestructurados que el proyecto necesitará en fases posteriores.

**Referencias (APA 7):**
Armbrust, M., Ghodsi, A., Xin, R., & Zaharia, M. (2021). 
Databricks. (2024). 
    

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo70"   # TODO: reemplazar NN
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
# TODO: ingerir las tablas del caso a la capa bronce, con esquema explícito
from pyspark.sql import functions as F

FUENTE = "samples.wanderbricks"

def ingerir_bronce(nombre_tabla, esquema_explicito, nombre_bronce=None):
    nombre_bronce = nombre_bronce or f"bronze_{nombre_tabla}"
    df_raw = spark.table(f"{FUENTE}.{nombre_tabla}")

    exprs = [F.col(c).cast(t).alias(c) for c, t in esquema_explicito.items()]
    df_tipado = df_raw.select(*exprs)

    df_bronce = (df_tipado
        .withColumn("_source_table", F.lit(f"{FUENTE}.{nombre_tabla}"))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (df_bronce.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.{nombre_bronce}"))

    print(f"✔ {nombre_bronce}: {df_bronce.count():,} filas ingeridas")


# --- Esquemas explícitos (nombres reales, confirmados por printSchema) ---

esquema_users = {
    "user_id":      "bigint",
    "email":        "string",
    "name":         "string",
    "country":      "string",
    "user_type":    "string",
    "created_at":   "timestamp",
    "is_business":  "boolean",
    "company_name": "string",
}

esquema_countries = {
    "country":      "string",
    "country_code": "string",
    "continent":    "string",
}

esquema_bookings = {
    "booking_id":   "bigint",
    "user_id":      "bigint",
    "property_id":  "bigint",
    "check_in":     "date",
    "check_out":    "date",
    "guests_count": "int",
    "total_amount": "float",
    "status":       "string",
    "created_at":   "timestamp",
    "updated_at":   "timestamp",
}

esquema_booking_updates = {
    "booking_update_id": "bigint",
    "booking_id":         "bigint",
    "user_id":            "bigint",
    "property_id":        "bigint",
    "check_in":           "date",
    "check_out":          "date",
    "guests_count":       "int",
    "total_amount":       "float",
    "status":             "string",
    "created_at":         "timestamp",
    "updated_at":         "timestamp",
}

# --- Ejecutar la ingesta ---
ingerir_bronce("users", esquema_users)
ingerir_bronce("countries", esquema_countries)
ingerir_bronce("bookings", esquema_bookings)
ingerir_bronce("booking_updates", esquema_booking_updates)

### 4.3 Capa plata — datos limpios y tipados

In [0]:
# TODO: limpieza, tipado y reglas de negocio

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -----------------------------------------------------------------
# 4.3 Capa plata — datos limpios y tipados
# -----------------------------------------------------------------

# --- silver_users: limpieza básica + normalización de country para el join ---
silver_users = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_users")
    .withColumn("country_clean", F.trim(F.lower(F.col("country"))))
    .filter(F.col("user_id").isNotNull())            # regla de negocio: sin user_id no sirve
    .dropDuplicates(["user_id"])                       # por si hay duplicados exactos
)

# --- silver_countries: misma normalización para que el join calce ---
silver_countries = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_countries")
    .withColumn("country_clean", F.trim(F.lower(F.col("country"))))
    .dropDuplicates(["country"])
)

# --- silver_bookings: reglas de negocio básicas de validez ---
silver_bookings = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_bookings")
    .filter(F.col("check_out") > F.col("check_in"))   # una reserva debe tener fechas coherentes
    .filter(F.col("total_amount") >= 0)                 # montos negativos no tienen sentido
)

# --- silver_booking_updates: historial completo, tipado, sin cambios de negocio ---
silver_booking_updates = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_booking_updates")

# --- silver_bookings_estado_actual: bookings + último estado conocido ---
w = Window.partitionBy("booking_id").orderBy(F.col("updated_at").desc())

ultimo_update = (silver_booking_updates
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumnRenamed("status", "status_actualizado")
    .withColumnRenamed("total_amount", "total_amount_actualizado")
    .withColumnRenamed("updated_at", "ultima_actualizacion")
    .select("booking_id", "status_actualizado", "total_amount_actualizado", "ultima_actualizacion")
)

silver_bookings_estado_actual = (silver_bookings
    .join(ultimo_update, on="booking_id", how="left")
    # si nunca hubo actualización, el estado vigente es el original de bookings
    .withColumn("status_vigente",
        F.coalesce(F.col("status_actualizado"), F.col("status")))
    .withColumn("monto_vigente",
        F.coalesce(F.col("total_amount_actualizado"), F.col("total_amount")))
)

# --- Escribir todas las tablas plata ---
tablas_plata = {
    "silver_users": silver_users,
    "silver_countries": silver_countries,
    "silver_bookings": silver_bookings,
    "silver_booking_updates": silver_booking_updates,
    "silver_bookings_estado_actual": silver_bookings_estado_actual,
}

for nombre, df in tablas_plata.items():
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.{nombre}"))
    print(f"✔ {nombre}: {df.count():,} filas")

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
# TODO: leer y aplanar estructuras anidadas
from pyspark.sql import functions as F

# -----------------------------------------------------------------
# 4.4 Datos semiestructurados
# -----------------------------------------------------------------
# Construimos una tabla donde cada reserva anida, en un array de structs,
# el historial completo de sus actualizaciones. Esto modela de forma nativa
# la relación 1:N booking -> booking_updates, sin necesitar un join en 
# tiempo de consulta.

updates_agrupados = (silver_booking_updates
    .groupBy("booking_id")
    .agg(
        F.collect_list(
            F.struct(
                F.col("booking_update_id"),
                F.col("status").alias("nuevo_status"),
                F.col("total_amount").alias("nuevo_monto"),
                F.col("check_in").alias("nuevo_check_in"),
                F.col("check_out").alias("nuevo_check_out"),
                F.col("updated_at")
            )
        ).alias("historial_actualizaciones")
    )
)

silver_bookings_anidado = (silver_bookings
    .join(updates_agrupados, on="booking_id", how="left")
    .withColumn(
        "num_actualizaciones",
        F.when(F.col("historial_actualizaciones").isNotNull(),
               F.size(F.col("historial_actualizaciones"))).otherwise(0)
    )
)

(silver_bookings_anidado.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_bookings_anidado"))

print(f"✔ silver_bookings_anidado: {silver_bookings_anidado.count():,} filas")

# Verificación visual de la estructura anidada
silver_bookings_anidado.select(
    "booking_id", "status", "num_actualizaciones", "historial_actualizaciones"
).filter(F.col("num_actualizaciones") > 1).show(5, truncate=False)

silver_bookings_anidado.printSchema()

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
# ACID — una operación que modifique datos
# TODO

# --- ACID: operación atómica sobre silver_bookings_estado_actual ---
from pyspark.sql import functions as F

# Contamos cuántas reservas "pending" con check_in ya pasado hay antes del cambio
pendientes_vencidas = spark.sql(f"""
    SELECT COUNT(*) AS total
    FROM {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual
    WHERE status_vigente = 'pending' AND check_in < current_date()
""")
pendientes_vencidas.show()

# UPDATE atómico: Delta garantiza que esta operación se aplica completa o no se aplica
spark.sql(f"""
    UPDATE {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual
    SET status_vigente = 'expired'
    WHERE status_vigente = 'pending' AND check_in < current_date()
""")

# Verificamos el resultado
spark.sql(f"""
    SELECT status_vigente, COUNT(*) AS total
    FROM {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual
    GROUP BY status_vigente
    ORDER BY total DESC
""").show()

In [0]:
# Time travel
# display(spark.sql(f"DESCRIBE HISTORY {TABLA}"))
# TODO: consultar una versión anterior y comparar

# --- Time Travel ---

# Vemos el historial de versiones de la tabla
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual"))

In [0]:
# --- Comparar versión anterior (0) vs. versión actual (1) ---

df_version_0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")

print("--- Versión 0 (antes del UPDATE) ---")
df_version_0.groupBy("status_vigente").count().orderBy(F.col("count").desc()).show()

df_version_actual = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")

print("--- Versión actual (después del UPDATE) ---")
df_version_actual.groupBy("status_vigente").count().orderBy(F.col("count").desc()).show()

In [0]:
# Evolución de esquema con mergeSchema
# TODO

# --- Evolución de esquema ---

# Esquema ANTES del cambio
print("--- Esquema antes ---")
spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual").printSchema()

# Agregamos una columna nueva calculada
df_con_nueva_columna = (spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")
    .withColumn("duracion_noches", F.datediff(F.col("check_out"), F.col("check_in")))
)

# Escribimos con mergeSchema=true para evolucionar el esquema sin recrear la tabla
(df_con_nueva_columna.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual"))

# Esquema DESPUÉS del cambio
print("--- Esquema después ---")
spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual").printSchema()

# Confirmamos con DESCRIBE HISTORY que quedó registrada una nueva versión
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual"))

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

Version SQL

¿En que paises se concentran mas las cancelaciones o reservas vencidas sin gestionar?
Esto le sirve al negocio para detectar mercados con politicas de rembolso o comunicacion deficientes

In [0]:
# Consulta 1 — pregunta que responde:
# TODO

spark.sql(f"""
SELECT
    c.country,
    c.continent,
    COUNT(*) AS total_reservas,
    SUM(CASE WHEN b.status_vigente = 'cancelled' THEN 1 ELSE 0 END) AS canceladas,
    SUM(CASE WHEN b.status_vigente = 'expired' THEN 1 ELSE 0 END) AS vencidas,
    ROUND(100.0 * SUM(CASE WHEN b.status_vigente IN ('cancelled','expired') THEN 1 ELSE 0 END) / COUNT(*), 2) AS tasa_problema_pct
FROM {CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual b
JOIN {CATALOGO}.{ESQUEMA}.silver_users u
    ON b.user_id = u.user_id
JOIN {CATALOGO}.{ESQUEMA}.silver_countries c
    ON TRIM(LOWER(u.country)) = TRIM(LOWER(c.country))
GROUP BY c.country, c.continent
HAVING COUNT(*) >= 30
ORDER BY tasa_problema_pct DESC
LIMIT 15
""").show(15, truncate=False)

Versión PySpark


In [0]:
from pyspark.sql import functions as F

df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings_estado_actual")
df_users = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_users")
df_countries = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_countries")

resultado_q1 = (df_bookings
    .join(df_users, on="user_id")
    .join(
        df_countries,
        on=F.trim(F.lower(df_users["country"])) == F.trim(F.lower(df_countries["country"])),
        how="inner"
    )
    .groupBy(df_countries["country"], df_countries["continent"])
    .agg(
        F.count("*").alias("total_reservas"),
        F.sum(F.when(F.col("status_vigente") == "cancelled", 1).otherwise(0)).alias("canceladas"),
        F.sum(F.when(F.col("status_vigente") == "expired", 1).otherwise(0)).alias("vencidas"),
        F.round(
            100.0 * F.sum(F.when(F.col("status_vigente").isin("cancelled", "expired"), 1).otherwise(0))
            / F.count("*"), 2
        ).alias("tasa_problema_pct")
    )
    .filter(F.col("total_reservas") >= 30)
    .orderBy(F.col("tasa_problema_pct").desc())
)

resultado_q1.show(15, truncate=False)

Version SQL

¿Cuanto tarda en promedio un usuario o el sistema en hacer el primer cambio sobre una reserva recien creada?
Esto ayuda a dimensionar si el proceso de confirmacion es lento y a priorizar automatizacion

In [0]:
spark.sql(f"""
WITH primera_actualizacion AS (
    SELECT
        booking_id,
        MIN(updated_at) AS primera_actualizacion
    FROM {CATALOGO}.{ESQUEMA}.silver_booking_updates
    GROUP BY booking_id
)
SELECT
    ROUND(AVG(TIMESTAMPDIFF(HOUR, b.created_at, p.primera_actualizacion)), 2) AS horas_promedio,
    ROUND(MIN(TIMESTAMPDIFF(HOUR, b.created_at, p.primera_actualizacion)), 2) AS horas_min,
    ROUND(MAX(TIMESTAMPDIFF(HOUR, b.created_at, p.primera_actualizacion)), 2) AS horas_max,
    COUNT(*) AS reservas_con_actualizacion
FROM {CATALOGO}.{ESQUEMA}.silver_bookings b
JOIN primera_actualizacion p
    ON b.booking_id = p.booking_id
""").show(truncate=False)

Versión PySpark

In [0]:
from pyspark.sql import functions as F

df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings")
df_updates = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_booking_updates")

primera_actualizacion = (df_updates
    .groupBy("booking_id")
    .agg(F.min("updated_at").alias("primera_actualizacion"))
)

resultado_q2 = (df_bookings
    .join(primera_actualizacion, on="booking_id")
    .withColumn(
        "horas_hasta_actualizacion",
        (F.col("primera_actualizacion").cast("long") - F.col("created_at").cast("long")) / 3600
    )
    .agg(
        F.round(F.avg("horas_hasta_actualizacion"), 2).alias("horas_promedio"),
        F.round(F.min("horas_hasta_actualizacion"), 2).alias("horas_min"),
        F.round(F.max("horas_hasta_actualizacion"), 2).alias("horas_max"),
        F.count("*").alias("reservas_con_actualizacion")
    )
)

resultado_q2.show(truncate=False)

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas